# 04 — Reverse Validation of Benchmark Completeness

This notebook performs a reverse-triangulation check of the Dutch official-source PEP benchmark snapshot exported by notebook 02.

The primary analysis evaluates OpenSanctions against the independently constructed benchmark. This notebook performs the reverse direction: it identifies Netherlands-assigned OpenSanctions records that are not linked to the benchmark and classifies them as possible candidates for further review.

OpenSanctions is not treated as ground truth. An OpenSanctions record not represented in the benchmark may reflect:

- a genuine benchmark omission;
- a person outside the defined empirical scope;
- an outdated or historical OpenSanctions record;
- a different role or institutional category;
- a name variation not resolved by the implemented matching procedure;
- insufficient information for classification.

The purpose is therefore to assess the robustness of the benchmark construction process and document potential limitations, not to claim definitive completeness. All reverse-validation results are specific to the current official-source benchmark snapshot.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import re
import unicodedata
import json

# -------------------------------------------------------------------
# Project folder setup
# -------------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

INPUT_DIR = PROJECT_DIR / "data" / "input"
EXTERNAL_DIR = PROJECT_DIR / "data" / "external"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# Input files from notebooks 02 and 03
# -------------------------------------------------------------------

BENCHMARK_PATH = (
    OUTPUT_DIR / "main_current_benchmark_v3_extended.csv"
)

MATCH_RESULTS_PATH = (
    OUTPUT_DIR / "benchmark_v3_opensanctions_match_results.csv"
)

OPENSANCTIONS_PATH = (
    EXTERNAL_DIR / "opensanctions_targets.simple.csv"
)

TAXONOMY_PATH = (
    INPUT_DIR / "taxonomy_v2_eu_aligned.xlsx"
)

# -------------------------------------------------------------------
# Reverse-validation outputs
# -------------------------------------------------------------------

REVERSE_CANDIDATES_PATH = (
    OUTPUT_DIR
    / "opensanctions_reverse_validation_candidates.csv"
)

REVERSE_CLASSIFICATION_TEMPLATE_PATH = (
    OUTPUT_DIR
    / "opensanctions_reverse_validation_classification_template.csv"
)

REVERSE_VALIDATION_SUMMARY_PATH = (
    OUTPUT_DIR
    / "opensanctions_reverse_validation_summary.csv"
)

print("Project folder:", PROJECT_DIR)

print("\nRequired input files:")
print("Final benchmark:", BENCHMARK_PATH.exists())
print("Match results:", MATCH_RESULTS_PATH.exists())
print("OpenSanctions export:", OPENSANCTIONS_PATH.exists())
print("Taxonomy workbook:", TAXONOMY_PATH.exists())

Project folder: c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean

Required input files:
Final benchmark: True
Match results: True
OpenSanctions export: True
Taxonomy workbook: True


## 1. Load the benchmark, matching results, and OpenSanctions export

The matching results identify OpenSanctions records that have already been linked to the benchmark through strict or rule-based matching.

The reverse-validation procedure focuses on Dutch-relevant OpenSanctions person records that remain unlinked after the completed matching procedure.

In [2]:
# -------------------------------------------------------------------
# File-loading and file-saving helpers
# -------------------------------------------------------------------

def read_csv_with_fallback_encodings(file_path):
    """
    Load a CSV file using common encodings.
    """
    possible_encodings = [
        "utf-8",
        "utf-8-sig",
        "cp1252",
        "latin1"
    ]

    last_error = None

    for encoding in possible_encodings:
        try:
            dataframe = pd.read_csv(
                file_path,
                encoding=encoding,
                low_memory=False
            )

            print(f"Loaded {file_path.name} using: {encoding}")

            return dataframe, encoding

        except UnicodeDecodeError as error:
            last_error = error

    raise ValueError(
        f"Could not load {file_path.name}. "
        f"Last error: {last_error}"
    )


def write_csv_safely(dataframe, output_path):
    """
    Save a UTF-8 CSV file.

    If the intended output is open in Excel, save a timestamped
    fallback file instead.
    """
    try:
        dataframe.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    except PermissionError:
        timestamp = datetime.now(
            timezone.utc
        ).strftime("%Y%m%d_%H%M%S")

        fallback_path = output_path.with_name(
            f"{output_path.stem}_{timestamp}{output_path.suffix}"
        )

        dataframe.to_csv(
            fallback_path,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "Warning: output file was locked. "
            f"Saved fallback file: {fallback_path.name}"
        )

        return fallback_path


# -------------------------------------------------------------------
# Load input datasets
# -------------------------------------------------------------------

benchmark_df, benchmark_encoding = (
    read_csv_with_fallback_encodings(BENCHMARK_PATH)
)

match_results_df, match_results_encoding = (
    read_csv_with_fallback_encodings(MATCH_RESULTS_PATH)
)

opensanctions_df, opensanctions_encoding = (
    read_csv_with_fallback_encodings(OPENSANCTIONS_PATH)
)

nl_taxonomy_df = pd.read_excel(
    TAXONOMY_PATH,
    sheet_name="nl_source_mapping_detail"
)

print("\nLoaded row counts:")
print("Benchmark records:", len(benchmark_df))
print("Matching-result records:", len(match_results_df))
print("OpenSanctions records:", len(opensanctions_df))
print("Dutch taxonomy rows:", len(nl_taxonomy_df))

print("\nMatch-result status distribution:")
display(
    match_results_df["final_match_status"]
    .value_counts(dropna=False)
    .rename_axis("final_match_status")
    .reset_index(name="record_count")
)

Loaded main_current_benchmark_v3_extended.csv using: utf-8
Loaded benchmark_v3_opensanctions_match_results.csv using: utf-8
Loaded opensanctions_targets.simple.csv using: utf-8

Loaded row counts:
Benchmark records: 675
Matching-result records: 675
OpenSanctions records: 753980
Dutch taxonomy rows: 16

Match-result status distribution:


,final_match_status,record_count
0,not_identified_by_implemented_matching,385
1,confirmed_automatic_match,277
2,probable_rule_based_match,8
3,ambiguous_exact_name_candidate,5


## 2. Restrict reverse validation to Netherlands-assigned OpenSanctions entities

The reverse-validation analysis is restricted to OpenSanctions person records explicitly assigned to the Netherlands in the `countries` field.

This restriction is appropriate for reverse validation because the purpose is to inspect the composition of the Dutch OpenSanctions population that is not linked to the official-source benchmark.

The filter is not used in the primary benchmark-to-OpenSanctions coverage analysis in notebook 03. Dutch PEPs may have foreign, multiple, or missing country assignments, particularly in diplomatic roles.

In [3]:
# -------------------------------------------------------------------
# Select OpenSanctions person entities explicitly assigned to NL
# -------------------------------------------------------------------

def has_netherlands_country_assignment(country_value):
    """
    Return True when the OpenSanctions countries field contains the
    Netherlands country code as a separate value.

    Examples:
    - 'nl' -> True
    - 'nl;be' -> True
    - '["nl", "be"]' -> True
    - NaN -> False
    """
    if pd.isna(country_value):
        return False

    country_text = str(country_value).strip().lower()

    if country_text == "":
        return False

    country_codes = re.findall(
        r"\b[a-z]{2,3}\b",
        country_text
    )

    return (
        "nl" in country_codes
        or "nld" in country_codes
    )


opensanctions_people_df = opensanctions_df[
    opensanctions_df["schema"]
    .astype("string")
    .eq("Person")
    .fillna(False)
].copy()

nl_opensanctions_people_df = opensanctions_people_df[
    opensanctions_people_df["countries"]
    .apply(has_netherlands_country_assignment)
].copy()

print("All OpenSanctions person records:", len(opensanctions_people_df))
print(
    "OpenSanctions person records explicitly assigned to NL:",
    len(nl_opensanctions_people_df)
)

print("\nExamples of Netherlands-assigned OpenSanctions records:")
display(
    nl_opensanctions_people_df[
        [
            "id",
            "name",
            "aliases",
            "countries",
            "dataset",
            "last_seen"
        ]
    ].head(20)
)

All OpenSanctions person records: 753980
OpenSanctions person records explicitly assigned to NL: 2893

Examples of Netherlands-assigned OpenSanctions records:


,id,name,aliases,countries,dataset,last_seen
1470,Q100308123,Mirjam Blaak,NaN,nl;ug,Wikidata Persons in Relevant Categories;Wikida...,2026-06-04T08:46:59
1686,Q100599330,Alida Francis,NaN,be;nl,PEP position annotations by OpenSanctions;Wiki...,2026-06-04T14:31:22
1823,Q100779990,Jan Keunen,NaN,nl,Wikidata Persons in Relevant Categories;Wikida...,2026-06-04T08:46:59
2119,Q100957674,Hilbert Bredemeijer,NaN,nl,Wikidata Persons in Relevant Categories;Wikida...,2026-06-04T08:46:59
2178,Q100989380,Greet Buter,NaN,nl,PEP position annotations by OpenSanctions;Wiki...,2026-06-04T14:31:22
2188,Q100997866,Robert van Asten,NaN,nl,European Commitee of the Regions Members;Nethe...,2026-06-04T14:31:22
2189,Q100997869,Martijn Balster,NaN,nl,Wikidata Persons in Relevant Categories;Wikida...,2026-06-04T08:46:59
2257,Q101072169,Danny de Vries,NaN,nl,PEP position annotations by OpenSanctions;Wiki...,2026-06-04T14:31:22
2505,Q101423101,Anke van Extel-van Katwijk,NaN,nl,PEP position annotations by OpenSanctions;Wiki...,2026-06-04T14:31:22
2608,Q101579497,Vincent Karremans,NaN,nl,PEP position annotations by OpenSanctions;US C...,2026-06-04T14:31:22


## 3. Reverse deterministic matching against the benchmark

The Netherlands-assigned OpenSanctions entities are matched against the final benchmark using the same name-comparison rules as notebook 03, applied in the reverse direction:

1. normalised exact matching;
2. cleaned exact matching;
3. unique initial–surname matching.

The reverse analysis is restricted to OpenSanctions Person entities whose `countries` field contains a Netherlands assignment. Fuzzy similarity remains a review aid and is not counted as a match.

In [4]:
# -------------------------------------------------------------------
# Name normalisation helpers
# -------------------------------------------------------------------

def normalise_for_matching(value):
    """
    Create a comparison-friendly key by removing accents, punctuation,
    and repeated whitespace.
    """
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()

    text = unicodedata.normalize("NFKD", text)

    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def create_clean_exact_key(value):
    """
    Create a cleaned exact-match key.

    This removes common titles, degrees, and parenthetical first-name
    variants, while preserving the remaining name structure.
    """
    if pd.isna(value):
        return ""

    raw_name = str(value).strip()

    parentheses_match = re.search(
        r"\(([^)]+)\)",
        raw_name
    )

    if parentheses_match:
        first_name_in_parentheses = (
            parentheses_match.group(1)
            .strip()
        )

        remaining_text = (
            raw_name[parentheses_match.end():]
            .strip()
        )

        raw_name = (
            f"{first_name_in_parentheses} {remaining_text}"
        ).strip()

    name_key = normalise_for_matching(raw_name)

    removable_tokens = {
        "mr",
        "dr",
        "drs",
        "prof",
        "professor",
        "ir",
        "ing",
        "ma",
        "mba",
        "msc",
        "bsc",
        "ll",
        "llm",
        "koning",
        "king"
    }

    cleaned_tokens = [
        token
        for token in name_key.split()
        if token not in removable_tokens
    ]

    return " ".join(cleaned_tokens).strip()


def split_aliases(alias_value):
    """
    Convert an OpenSanctions aliases value into a list of names.
    """
    if pd.isna(alias_value):
        return []

    alias_text = str(alias_value).strip()

    if alias_text == "":
        return []

    try:
        parsed_aliases = json.loads(alias_text)

        if isinstance(parsed_aliases, list):
            return [
                str(alias).strip()
                for alias in parsed_aliases
                if str(alias).strip()
            ]

        if isinstance(parsed_aliases, str):
            alias_text = parsed_aliases

    except (
        json.JSONDecodeError,
        TypeError
    ):
        pass

    return [
        alias.strip()
        for alias in re.split(
            r"[;|]",
            alias_text
        )
        if alias.strip()
    ]


def extract_initial_surname_features(name_value):
    """
    Extract leading initials and the complete surname component.

    Example:
    'I.J. Visseren-Hamakers'
    -> first initial: i
    -> surname key: visseren hamakers
    """
    name_key = normalise_for_matching(name_value)

    if name_key == "":
        return {
            "leading_initials": [],
            "surname_key": "",
            "first_initial": ""
        }

    tokens = name_key.split()

    initials = []
    token_position = 0

    while (
        token_position < len(tokens)
        and re.fullmatch(
            r"[a-z]",
            tokens[token_position]
        )
    ):
        initials.append(tokens[token_position])
        token_position += 1

    surname_tokens = tokens[token_position:]

    if len(initials) == 0 or len(surname_tokens) == 0:
        return {
            "leading_initials": [],
            "surname_key": "",
            "first_initial": ""
        }

    return {
        "leading_initials": initials,
        "surname_key": " ".join(surname_tokens),
        "first_initial": initials[0]
    }


def candidate_ends_with_full_surname(
    candidate_name_key,
    benchmark_surname_key
):
    """
    Check whether the candidate name ends with the benchmark's full
    surname component, including compound surnames.
    """
    if (
        candidate_name_key == ""
        or benchmark_surname_key == ""
    ):
        return False

    return (
        candidate_name_key == benchmark_surname_key
        or candidate_name_key.endswith(
            f" {benchmark_surname_key}"
        )
    )


def candidate_first_initial(candidate_name_key):
    """
    Return the first alphabetic initial from a cleaned candidate key.
    """
    if pd.isna(candidate_name_key):
        return ""

    tokens = str(candidate_name_key).split()

    if len(tokens) == 0:
        return ""

    return tokens[0][0]


# -------------------------------------------------------------------
# Prepare benchmark person-level lookup tables
# -------------------------------------------------------------------
# The same benchmark person may theoretically occur in more than one
# category. We retain all related record IDs as audit information.

benchmark_reverse_df = benchmark_df[
    [
        "benchmark_record_id",
        "person_name",
        "taxonomy_category",
        "role_title",
        "institution"
    ]
].copy()

benchmark_reverse_df["strict_name_key"] = (
    benchmark_reverse_df["person_name"]
    .apply(normalise_for_matching)
)

benchmark_reverse_df["clean_exact_key"] = (
    benchmark_reverse_df["person_name"]
    .apply(create_clean_exact_key)
)

benchmark_reverse_df["benchmark_person_key"] = (
    benchmark_reverse_df["strict_name_key"]
)

benchmark_person_lookup_df = (
    benchmark_reverse_df
    .groupby(
        [
            "benchmark_person_key",
            "strict_name_key",
            "clean_exact_key"
        ],
        dropna=False
    )
    .agg(
        benchmark_person_name=(
            "person_name",
            "first"
        ),
        benchmark_record_ids=(
            "benchmark_record_id",
            lambda values: "; ".join(
                str(value)
                for value in sorted(set(values))
            )
        ),
        benchmark_categories=(
            "taxonomy_category",
            lambda values: "; ".join(
                sorted(set(values))
            )
        ),
        benchmark_roles=(
            "role_title",
            lambda values: "; ".join(
                sorted(
                    set(
                        str(value)
                        for value in values
                        if pd.notna(value)
                    )
                )
            )
        )
    )
    .reset_index()
)

print("Unique benchmark persons:", len(benchmark_person_lookup_df))


# -------------------------------------------------------------------
# Create primary-name and alias-name rows for NL OpenSanctions entities
# -------------------------------------------------------------------

nl_primary_names_df = nl_opensanctions_people_df[
    [
        "id",
        "name",
        "countries",
        "dataset",
        "first_seen",
        "last_seen",
        "last_change"
    ]
].copy()

nl_primary_names_df = nl_primary_names_df.rename(
    columns={
        "id": "opensanctions_id",
        "name": "opensanctions_name"
    }
)

nl_primary_names_df["opensanctions_name_type"] = "primary_name"

nl_alias_names_df = nl_opensanctions_people_df[
    [
        "id",
        "aliases",
        "countries",
        "dataset",
        "first_seen",
        "last_seen",
        "last_change"
    ]
].copy()

nl_alias_names_df = nl_alias_names_df[
    nl_alias_names_df["aliases"].notna()
].copy()

nl_alias_names_df["opensanctions_name"] = (
    nl_alias_names_df["aliases"]
    .apply(split_aliases)
)

nl_alias_names_df = (
    nl_alias_names_df
    .explode("opensanctions_name")
    .dropna(subset=["opensanctions_name"])
    .copy()
)

nl_alias_names_df = nl_alias_names_df[
    nl_alias_names_df["opensanctions_name"]
    .astype("string")
    .str.strip()
    .ne("")
].copy()

nl_alias_names_df = nl_alias_names_df.rename(
    columns={
        "id": "opensanctions_id"
    }
)

nl_alias_names_df["opensanctions_name_type"] = "alias"

nl_alias_names_df = nl_alias_names_df[
    [
        "opensanctions_id",
        "opensanctions_name",
        "opensanctions_name_type",
        "countries",
        "dataset",
        "first_seen",
        "last_seen",
        "last_change"
    ]
].copy()

nl_os_names_df = pd.concat(
    [
        nl_primary_names_df,
        nl_alias_names_df
    ],
    ignore_index=True
)

nl_os_names_df["strict_name_key"] = (
    nl_os_names_df["opensanctions_name"]
    .apply(normalise_for_matching)
)

nl_os_names_df["clean_exact_key"] = (
    nl_os_names_df["opensanctions_name"]
    .apply(create_clean_exact_key)
)

nl_os_names_df["name_type_rank"] = (
    nl_os_names_df["opensanctions_name_type"]
    .map(
        {
            "primary_name": 0,
            "alias": 1
        }
    )
    .fillna(9)
)

nl_os_names_df = (
    nl_os_names_df[
        nl_os_names_df["strict_name_key"]
        .astype("string")
        .str.len()
        .gt(0)
    ]
    .drop_duplicates(
        subset=[
            "opensanctions_id",
            "opensanctions_name",
            "opensanctions_name_type"
        ]
    )
    .reset_index(drop=True)
)

print("NL OpenSanctions primary-name rows:", len(nl_primary_names_df))
print("NL OpenSanctions alias-name rows:", len(nl_alias_names_df))
print("NL OpenSanctions total name rows:", len(nl_os_names_df))

Unique benchmark persons: 651
NL OpenSanctions primary-name rows: 2893
NL OpenSanctions alias-name rows: 382
NL OpenSanctions total name rows: 3103


### Apply strict, cleaned, and initial–surname matching

Each Netherlands-assigned OpenSanctions entity receives one reverse-match status.

An entity is considered linked to the benchmark when at least one of its primary-name or alias-name representations identifies one benchmark person uniquely through a deterministic matching rule.

In [5]:
# -------------------------------------------------------------------
# Reverse matching: strict normalised exact matching
# -------------------------------------------------------------------

REVERSE_MATCH_RESULTS_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_reverse_match_results.csv"
)

REVERSE_MATCH_PAIRS_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_reverse_match_pairs.csv"
)

strict_reverse_links_df = nl_os_names_df.merge(
    benchmark_person_lookup_df[
        [
            "benchmark_person_key",
            "strict_name_key",
            "benchmark_person_name",
            "benchmark_record_ids",
            "benchmark_categories",
            "benchmark_roles"
        ]
    ],
    on="strict_name_key",
    how="inner"
)

strict_reverse_summary_df = (
    strict_reverse_links_df
    .groupby("opensanctions_id")
    .agg(
        strict_benchmark_person_count=(
            "benchmark_person_key",
            "nunique"
        ),
        strict_benchmark_person_names=(
            "benchmark_person_name",
            lambda values: "; ".join(
                sorted(set(values))
            )
        ),
        strict_benchmark_record_ids=(
            "benchmark_record_ids",
            lambda values: "; ".join(
                sorted(set(values))
            )
        ),
        strict_benchmark_categories=(
            "benchmark_categories",
            lambda values: "; ".join(
                sorted(set(values))
            )
        )
    )
    .reset_index()
)

# -------------------------------------------------------------------
# Create one entity-level table for every NL OpenSanctions person
# -------------------------------------------------------------------

nl_os_entity_df = nl_opensanctions_people_df[
    [
        "id",
        "name",
        "aliases",
        "countries",
        "dataset",
        "birth_date",
        "first_seen",
        "last_seen",
        "last_change"
    ]
].copy()

nl_os_entity_df = nl_os_entity_df.rename(
    columns={
        "id": "opensanctions_id",
        "name": "opensanctions_primary_name"
    }
)

reverse_results_df = nl_os_entity_df.merge(
    strict_reverse_summary_df,
    on="opensanctions_id",
    how="left"
)

reverse_results_df["strict_benchmark_person_count"] = (
    reverse_results_df["strict_benchmark_person_count"]
    .fillna(0)
    .astype(int)
)

reverse_results_df["reverse_match_type"] = (
    "no_deterministic_match"
)

reverse_results_df.loc[
    reverse_results_df["strict_benchmark_person_count"].eq(1),
    "reverse_match_type"
] = "normalised_exact"

reverse_results_df.loc[
    reverse_results_df["strict_benchmark_person_count"].gt(1),
    "reverse_match_type"
] = "normalised_exact_ambiguous"


# -------------------------------------------------------------------
# Reverse matching: cleaned exact matching
# -------------------------------------------------------------------
# This stage is applied only when strict matching found no benchmark
# person candidate.

clean_reverse_links_df = nl_os_names_df.merge(
    benchmark_person_lookup_df[
        [
            "benchmark_person_key",
            "clean_exact_key",
            "benchmark_person_name",
            "benchmark_record_ids",
            "benchmark_categories",
            "benchmark_roles"
        ]
    ],
    on="clean_exact_key",
    how="inner"
)

clean_reverse_summary_df = (
    clean_reverse_links_df
    .groupby("opensanctions_id")
    .agg(
        clean_benchmark_person_count=(
            "benchmark_person_key",
            "nunique"
        ),
        clean_benchmark_person_names=(
            "benchmark_person_name",
            lambda values: "; ".join(
                sorted(set(values))
            )
        ),
        clean_benchmark_record_ids=(
            "benchmark_record_ids",
            lambda values: "; ".join(
                sorted(set(values))
            )
        ),
        clean_benchmark_categories=(
            "benchmark_categories",
            lambda values: "; ".join(
                sorted(set(values))
            )
        )
    )
    .reset_index()
)

reverse_results_df = reverse_results_df.merge(
    clean_reverse_summary_df,
    on="opensanctions_id",
    how="left"
)

reverse_results_df["clean_benchmark_person_count"] = (
    reverse_results_df["clean_benchmark_person_count"]
    .fillna(0)
    .astype(int)
)

clean_unique_mask = (
    reverse_results_df["strict_benchmark_person_count"].eq(0)
    & reverse_results_df["clean_benchmark_person_count"].eq(1)
)

clean_ambiguous_mask = (
    reverse_results_df["strict_benchmark_person_count"].eq(0)
    & reverse_results_df["clean_benchmark_person_count"].gt(1)
)

reverse_results_df.loc[
    clean_unique_mask,
    "reverse_match_type"
] = "cleaned_exact"

reverse_results_df.loc[
    clean_ambiguous_mask,
    "reverse_match_type"
] = "cleaned_exact_ambiguous"


# -------------------------------------------------------------------
# Reverse matching: unique initial–surname rule
# -------------------------------------------------------------------
# This reproduces the abbreviation-aware logic from notebook 03.
# It starts with benchmark names containing initials and searches for
# one unique NL OpenSanctions entity with matching full surname and
# first initial.

benchmark_initial_df = (
    benchmark_person_lookup_df[
        [
            "benchmark_person_key",
            "benchmark_person_name",
            "benchmark_record_ids",
            "benchmark_categories",
            "benchmark_roles"
        ]
    ]
    .copy()
)

benchmark_initial_features_df = (
    benchmark_initial_df["benchmark_person_name"]
    .apply(extract_initial_surname_features)
    .apply(pd.Series)
)

benchmark_initial_df = pd.concat(
    [
        benchmark_initial_df.reset_index(drop=True),
        benchmark_initial_features_df.reset_index(drop=True)
    ],
    axis=1
)

benchmark_initial_df = benchmark_initial_df[
    benchmark_initial_df["surname_key"]
    .astype("string")
    .str.len()
    .gt(0)
].copy()

nl_os_names_initial_df = nl_os_names_df.copy()

nl_os_names_initial_df["candidate_first_initial"] = (
    nl_os_names_initial_df["clean_exact_key"]
    .apply(candidate_first_initial)
)

initial_surname_pair_rows = []

for _, benchmark_row in benchmark_initial_df.iterrows():

    benchmark_surname_key = benchmark_row["surname_key"]
    benchmark_first_initial = benchmark_row["first_initial"]

    candidate_rows_df = nl_os_names_initial_df[
        nl_os_names_initial_df["candidate_first_initial"]
        .eq(benchmark_first_initial)
    ].copy()

    candidate_rows_df = candidate_rows_df[
        candidate_rows_df["clean_exact_key"]
        .apply(
            lambda candidate_key: (
                candidate_ends_with_full_surname(
                    candidate_name_key=candidate_key,
                    benchmark_surname_key=benchmark_surname_key
                )
            )
        )
    ].copy()

    candidate_ids = (
        candidate_rows_df["opensanctions_id"]
        .dropna()
        .unique()
        .tolist()
    )

    # The rule accepts only one unique OpenSanctions entity.
    if len(candidate_ids) != 1:
        continue

    matched_opensanctions_id = candidate_ids[0]

    selected_name_df = (
        candidate_rows_df[
            candidate_rows_df["opensanctions_id"]
            .eq(matched_opensanctions_id)
        ]
        .sort_values(
            [
                "name_type_rank",
                "opensanctions_name"
            ]
        )
        .head(1)
    )

    selected_name_row = selected_name_df.iloc[0]

    initial_surname_pair_rows.append({
        "opensanctions_id": matched_opensanctions_id,
        "benchmark_person_key": benchmark_row[
            "benchmark_person_key"
        ],
        "benchmark_person_name": benchmark_row[
            "benchmark_person_name"
        ],
        "benchmark_record_ids": benchmark_row[
            "benchmark_record_ids"
        ],
        "benchmark_categories": benchmark_row[
            "benchmark_categories"
        ],
        "benchmark_surname_key": benchmark_surname_key,
        "benchmark_first_initial": benchmark_first_initial,
        "matched_opensanctions_name": selected_name_row[
            "opensanctions_name"
        ],
        "matched_opensanctions_name_type": selected_name_row[
            "opensanctions_name_type"
        ],
        "reverse_match_type": "initial_surname_unique"
    })

initial_surname_reverse_pairs_df = pd.DataFrame(
    initial_surname_pair_rows
)

initial_surname_reverse_summary_df = (
    initial_surname_reverse_pairs_df
    .groupby("opensanctions_id")
    .agg(
        initial_surname_benchmark_person_count=(
            "benchmark_person_key",
            "nunique"
        ),
        initial_surname_benchmark_person_names=(
            "benchmark_person_name",
            lambda values: "; ".join(
                sorted(set(values))
            )
        ),
        initial_surname_benchmark_record_ids=(
            "benchmark_record_ids",
            lambda values: "; ".join(
                sorted(set(values))
            )
        ),
        initial_surname_benchmark_categories=(
            "benchmark_categories",
            lambda values: "; ".join(
                sorted(set(values))
            )
        )
    )
    .reset_index()
)

reverse_results_df = reverse_results_df.merge(
    initial_surname_reverse_summary_df,
    on="opensanctions_id",
    how="left"
)

reverse_results_df[
    "initial_surname_benchmark_person_count"
] = (
    reverse_results_df[
        "initial_surname_benchmark_person_count"
    ]
    .fillna(0)
    .astype(int)
)

initial_surname_unique_mask = (
    reverse_results_df["reverse_match_type"]
    .eq("no_deterministic_match")
    & reverse_results_df[
        "initial_surname_benchmark_person_count"
    ].eq(1)
)

reverse_results_df.loc[
    initial_surname_unique_mask,
    "reverse_match_type"
] = "initial_surname_unique"


# -------------------------------------------------------------------
# Create unified linked-benchmark fields
# -------------------------------------------------------------------

reverse_results_df["linked_benchmark_person_names"] = pd.NA
reverse_results_df["linked_benchmark_record_ids"] = pd.NA
reverse_results_df["linked_benchmark_categories"] = pd.NA

strict_match_mask = (
    reverse_results_df["reverse_match_type"]
    .eq("normalised_exact")
)

clean_match_mask = (
    reverse_results_df["reverse_match_type"]
    .eq("cleaned_exact")
)

initial_match_mask = (
    reverse_results_df["reverse_match_type"]
    .eq("initial_surname_unique")
)

reverse_results_df.loc[
    strict_match_mask,
    "linked_benchmark_person_names"
] = reverse_results_df.loc[
    strict_match_mask,
    "strict_benchmark_person_names"
]

reverse_results_df.loc[
    strict_match_mask,
    "linked_benchmark_record_ids"
] = reverse_results_df.loc[
    strict_match_mask,
    "strict_benchmark_record_ids"
]

reverse_results_df.loc[
    strict_match_mask,
    "linked_benchmark_categories"
] = reverse_results_df.loc[
    strict_match_mask,
    "strict_benchmark_categories"
]

reverse_results_df.loc[
    clean_match_mask,
    "linked_benchmark_person_names"
] = reverse_results_df.loc[
    clean_match_mask,
    "clean_benchmark_person_names"
]

reverse_results_df.loc[
    clean_match_mask,
    "linked_benchmark_record_ids"
] = reverse_results_df.loc[
    clean_match_mask,
    "clean_benchmark_record_ids"
]

reverse_results_df.loc[
    clean_match_mask,
    "linked_benchmark_categories"
] = reverse_results_df.loc[
    clean_match_mask,
    "clean_benchmark_categories"
]

reverse_results_df.loc[
    initial_match_mask,
    "linked_benchmark_person_names"
] = reverse_results_df.loc[
    initial_match_mask,
    "initial_surname_benchmark_person_names"
]

reverse_results_df.loc[
    initial_match_mask,
    "linked_benchmark_record_ids"
] = reverse_results_df.loc[
    initial_match_mask,
    "initial_surname_benchmark_record_ids"
]

reverse_results_df.loc[
    initial_match_mask,
    "linked_benchmark_categories"
] = reverse_results_df.loc[
    initial_match_mask,
    "initial_surname_benchmark_categories"
]

reverse_results_df["reverse_match_confidence"] = "none"

reverse_results_df.loc[
    reverse_results_df["reverse_match_type"].isin(
        [
            "normalised_exact",
            "cleaned_exact"
        ]
    ),
    "reverse_match_confidence"
] = "high"

reverse_results_df.loc[
    reverse_results_df["reverse_match_type"]
    .eq("initial_surname_unique"),
    "reverse_match_confidence"
] = "high_rule_based"

reverse_results_df.loc[
    reverse_results_df["reverse_match_type"]
    .isin(
        [
            "normalised_exact_ambiguous",
            "cleaned_exact_ambiguous"
        ]
    ),
    "reverse_match_confidence"
] = "ambiguous"

# Save record-level reverse results before fuzzy triage.
reverse_match_results_output_path = write_csv_safely(
    reverse_results_df,
    REVERSE_MATCH_RESULTS_PATH
)

# Save all deterministic linked pairs for auditability.
reverse_match_pairs_df = pd.concat(
    [
        strict_reverse_links_df.assign(
            reverse_match_type="normalised_exact"
        ),
        clean_reverse_links_df.assign(
            reverse_match_type="cleaned_exact"
        ),
        initial_surname_reverse_pairs_df
    ],
    ignore_index=True,
    sort=False
)

reverse_match_pairs_output_path = write_csv_safely(
    reverse_match_pairs_df,
    REVERSE_MATCH_PAIRS_PATH
)

print("Saved reverse match results:")
print(reverse_match_results_output_path)

print("\nSaved reverse match pairs:")
print(reverse_match_pairs_output_path)

print("\nReverse deterministic matching summary:")
display(
    reverse_results_df["reverse_match_type"]
    .value_counts()
    .rename_axis("reverse_match_type")
    .reset_index(name="opensanctions_entities")
)

Saved reverse match results:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_reverse_match_results.csv

Saved reverse match pairs:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_reverse_match_pairs.csv

Reverse deterministic matching summary:


,reverse_match_type,opensanctions_entities
0,no_deterministic_match,2602
1,normalised_exact,273
2,cleaned_exact,11
3,initial_surname_unique,7


## 4. Review overview of Netherlands-assigned entities without a deterministic benchmark link

Entities without a deterministic link are not interpreted as benchmark omissions automatically.

For each unlinked OpenSanctions entity, the output includes:

- primary name and aliases;
- OpenSanctions source datasets;
- first and last observation timestamps;
- the best fuzzy benchmark candidate, if any;
- a preliminary review hint based on the dataset label;
- blank manual-classification fields.

The manual review fields allow records to be classified as a possible benchmark omission, out of scope, historical, a name variation, or unclear.

In [6]:
# -------------------------------------------------------------------
# Fuzzy triage for NL entities without deterministic matches
# -------------------------------------------------------------------
# Fuzzy scores are included only to support manual review. They are
# never counted as reverse benchmark matches automatically.

from rapidfuzz import fuzz, process

REVERSE_UNMATCHED_OVERVIEW_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_benchmark_overview.csv"
)

REVERSE_UNMATCHED_DATASET_SUMMARY_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_dataset_summary.csv"
)

REVERSE_CLASSIFICATION_TEMPLATE_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_reverse_classification_template.csv"
)


def get_last_name_token(name_key):
    """
    Return the final token of a normalised name.
    """
    if pd.isna(name_key) or str(name_key).strip() == "":
        return ""

    return str(name_key).strip().split()[-1]


def get_first_initial_from_key(name_key):
    """
    Return the first alphabetic character from a name key.
    """
    letters_only = re.sub(
        r"[^a-z]",
        "",
        str(name_key).lower()
    )

    return letters_only[0] if letters_only else ""


def fuzzy_review_priority(
    score,
    same_last_name,
    first_initial_matches
):
    """
    Assign a review priority only. This function never confirms a match.
    """
    if (
        score >= 95
        and same_last_name
        and first_initial_matches
    ):
        return "high_priority_review"

    if (
        score >= 88
        and same_last_name
    ):
        return "medium_priority_review"

    return "standard_review"


def create_preliminary_review_hint(dataset_value):
    """
    Provide a non-binding hint for manual review based on source labels.

    This is not a final classification and does not make factual claims
    about the person's current role or benchmark eligibility.
    """
    dataset_text = (
        ""
        if pd.isna(dataset_value)
        else str(dataset_value).lower()
    )

    if "state-owned enterprise" in dataset_text:
        return (
            "Possible excluded state-owned-enterprise category; "
            "review against the empirical scope decision."
        )

    if (
        "senate" in dataset_text
        or "house of representatives" in dataset_text
    ):
        return (
            "Possible historical member, source-timing difference, "
            "or unresolved name variation."
        )

    if "local government" in dataset_text:
        return (
            "Possible subnational or out-of-scope public function; "
            "review empirical taxonomy scope."
        )

    return (
        "Review role, timing, and empirical-scope eligibility."
    )


unmatched_nl_os_df = reverse_results_df[
    reverse_results_df["reverse_match_type"]
    .eq("no_deterministic_match")
].copy()

print(
    "NL OpenSanctions entities without a deterministic benchmark link:",
    len(unmatched_nl_os_df)
)

# -------------------------------------------------------------------
# Find one best fuzzy benchmark candidate for each unlinked entity
# -------------------------------------------------------------------

benchmark_fuzzy_lookup_df = benchmark_person_lookup_df[
    [
        "benchmark_person_key",
        "benchmark_person_name",
        "benchmark_record_ids",
        "benchmark_categories",
        "clean_exact_key"
    ]
].copy()

benchmark_fuzzy_lookup_df = benchmark_fuzzy_lookup_df[
    benchmark_fuzzy_lookup_df["clean_exact_key"]
    .astype("string")
    .str.len()
    .gt(0)
].copy()

benchmark_fuzzy_keys = (
    benchmark_fuzzy_lookup_df["clean_exact_key"]
    .tolist()
)

unmatched_os_name_rows_df = nl_os_names_df.merge(
    unmatched_nl_os_df[
        ["opensanctions_id"]
    ],
    on="opensanctions_id",
    how="inner"
)

best_fuzzy_rows = []

for opensanctions_id, entity_name_rows_df in (
    unmatched_os_name_rows_df
    .groupby("opensanctions_id")
):
    best_candidate = None

    for _, name_row in entity_name_rows_df.iterrows():
        os_name_key = name_row["clean_exact_key"]

        if os_name_key == "":
            continue

        fuzzy_result = process.extractOne(
            os_name_key,
            benchmark_fuzzy_keys,
            scorer=fuzz.WRatio
        )

        if fuzzy_result is None:
            continue

        _, score, benchmark_position = fuzzy_result

        benchmark_candidate_row = (
            benchmark_fuzzy_lookup_df
            .iloc[benchmark_position]
        )

        candidate_data = {
            "opensanctions_id": opensanctions_id,
            "best_fuzzy_source_name": name_row[
                "opensanctions_name"
            ],
            "best_fuzzy_source_name_type": name_row[
                "opensanctions_name_type"
            ],
            "best_fuzzy_score": round(float(score), 2),
            "best_fuzzy_benchmark_person_name": (
                benchmark_candidate_row[
                    "benchmark_person_name"
                ]
            ),
            "best_fuzzy_benchmark_record_ids": (
                benchmark_candidate_row[
                    "benchmark_record_ids"
                ]
            ),
            "best_fuzzy_benchmark_categories": (
                benchmark_candidate_row[
                    "benchmark_categories"
                ]
            )
        }

        if (
            best_candidate is None
            or candidate_data["best_fuzzy_score"]
            > best_candidate["best_fuzzy_score"]
        ):
            best_candidate = candidate_data

    if best_candidate is not None:
        best_fuzzy_rows.append(best_candidate)

best_fuzzy_reverse_df = pd.DataFrame(best_fuzzy_rows)

unmatched_nl_os_df = unmatched_nl_os_df.merge(
    best_fuzzy_reverse_df,
    on="opensanctions_id",
    how="left"
)

unmatched_nl_os_df["opensanctions_last_name"] = (
    unmatched_nl_os_df["opensanctions_primary_name"]
    .apply(normalise_for_matching)
    .apply(get_last_name_token)
)

unmatched_nl_os_df["benchmark_candidate_last_name"] = (
    unmatched_nl_os_df[
        "best_fuzzy_benchmark_person_name"
    ]
    .apply(normalise_for_matching)
    .apply(get_last_name_token)
)

unmatched_nl_os_df["same_last_name_as_fuzzy_candidate"] = (
    unmatched_nl_os_df["opensanctions_last_name"]
    .eq(
        unmatched_nl_os_df[
            "benchmark_candidate_last_name"
        ]
    )
)

unmatched_nl_os_df["source_first_initial"] = (
    unmatched_nl_os_df["opensanctions_primary_name"]
    .apply(normalise_for_matching)
    .apply(get_first_initial_from_key)
)

unmatched_nl_os_df["benchmark_candidate_first_initial"] = (
    unmatched_nl_os_df[
        "best_fuzzy_benchmark_person_name"
    ]
    .apply(normalise_for_matching)
    .apply(get_first_initial_from_key)
)

unmatched_nl_os_df[
    "first_initial_matches_fuzzy_candidate"
] = (
    unmatched_nl_os_df["source_first_initial"]
    .eq(
        unmatched_nl_os_df[
            "benchmark_candidate_first_initial"
        ]
    )
)

unmatched_nl_os_df["fuzzy_review_priority"] = (
    unmatched_nl_os_df.apply(
        lambda row: fuzzy_review_priority(
            score=(
                row["best_fuzzy_score"]
                if pd.notna(row["best_fuzzy_score"])
                else 0
            ),
            same_last_name=row[
                "same_last_name_as_fuzzy_candidate"
            ],
            first_initial_matches=row[
                "first_initial_matches_fuzzy_candidate"
            ]
        ),
        axis=1
    )
)

unmatched_nl_os_df["preliminary_review_hint"] = (
    unmatched_nl_os_df["dataset"]
    .apply(create_preliminary_review_hint)
)

# -------------------------------------------------------------------
# Add blank manual-review fields
# -------------------------------------------------------------------

unmatched_nl_os_df["manual_classification"] = (
    "needs_manual_review"
)

unmatched_nl_os_df["manual_scope_assessment"] = pd.NA
unmatched_nl_os_df["manual_role_or_category"] = pd.NA
unmatched_nl_os_df["manual_review_notes"] = pd.NA
unmatched_nl_os_df["reviewer"] = pd.NA
unmatched_nl_os_df["review_date"] = pd.NA

# -------------------------------------------------------------------
# Create a dataset-level overview for quick analysis
# -------------------------------------------------------------------

unmatched_dataset_memberships_df = unmatched_nl_os_df[
    [
        "opensanctions_id",
        "dataset",
        "fuzzy_review_priority"
    ]
].copy()

unmatched_dataset_memberships_df["dataset_label"] = (
    unmatched_dataset_memberships_df["dataset"]
    .fillna("Unknown dataset")
    .astype(str)
    .str.split(";")
)

unmatched_dataset_memberships_df = (
    unmatched_dataset_memberships_df
    .explode("dataset_label")
    .copy()
)

unmatched_dataset_memberships_df["dataset_label"] = (
    unmatched_dataset_memberships_df["dataset_label"]
    .astype(str)
    .str.strip()
)

unmatched_dataset_summary_df = (
    unmatched_dataset_memberships_df
    .groupby("dataset_label")
    .agg(
        unmatched_nl_entities=(
            "opensanctions_id",
            "nunique"
        ),
        high_priority_fuzzy_candidates=(
            "fuzzy_review_priority",
            lambda values: (
                values
                .eq("high_priority_review")
                .sum()
            )
        ),
        medium_priority_fuzzy_candidates=(
            "fuzzy_review_priority",
            lambda values: (
                values
                .eq("medium_priority_review")
                .sum()
            )
        )
    )
    .reset_index()
    .sort_values(
        "unmatched_nl_entities",
        ascending=False
    )
)

# -------------------------------------------------------------------
# Save reverse-validation outputs
# -------------------------------------------------------------------

unmatched_overview_output_path = write_csv_safely(
    unmatched_nl_os_df,
    REVERSE_UNMATCHED_OVERVIEW_PATH
)

classification_template_output_path = write_csv_safely(
    unmatched_nl_os_df,
    REVERSE_CLASSIFICATION_TEMPLATE_PATH
)

unmatched_dataset_summary_output_path = write_csv_safely(
    unmatched_dataset_summary_df,
    REVERSE_UNMATCHED_DATASET_SUMMARY_PATH
)

print("\nSaved unmatched NL OpenSanctions overview:")
print(unmatched_overview_output_path)

print("\nSaved manual classification template:")
print(classification_template_output_path)

print("\nSaved unmatched dataset summary:")
print(unmatched_dataset_summary_output_path)

print("\nUnmatched NL OpenSanctions entities by source dataset:")
display(unmatched_dataset_summary_df.head(30))

print("\nHighest-priority fuzzy review cases:")
display(
    unmatched_nl_os_df[
        [
            "opensanctions_id",
            "opensanctions_primary_name",
            "dataset",
            "best_fuzzy_score",
            "best_fuzzy_benchmark_person_name",
            "best_fuzzy_benchmark_categories",
            "fuzzy_review_priority",
            "preliminary_review_hint"
        ]
    ]
    .sort_values(
        [
            "fuzzy_review_priority",
            "best_fuzzy_score"
        ],
        ascending=[True, False]
    )
    .head(30)
)

print("\nReverse-validation overview:")
print(
    "NL OpenSanctions entities:",
    len(reverse_results_df)
)
print(
    "Linked through deterministic matching:",
    int(
        reverse_results_df["reverse_match_type"]
        .isin(
            [
                "normalised_exact",
                "cleaned_exact",
                "initial_surname_unique"
            ]
        )
        .sum()
    )
)
print(
    "Unlinked after deterministic matching:",
    len(unmatched_nl_os_df)
)

NL OpenSanctions entities without a deterministic benchmark link: 2602

Saved unmatched NL OpenSanctions overview:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_benchmark_overview.csv

Saved manual classification template:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_reverse_classification_template.csv

Saved unmatched dataset summary:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_dataset_summary.csv

Unmatched NL OpenSanctions entities by source dataset:


,dataset_label,unmatched_nl_entities,high_priority_fuzzy_candidates,medium_priority_fuzzy_candidates
11,Wikidata Persons in Relevant Categories,2479,2,3
12,Wikidata Politically Exposed Persons,2117,2,3
9,PEP position annotations by OpenSanctions,1613,1,3
7,MySociety's EveryPolitician Legislators,248,0,0
2,European Parliament Members,32,0,0
1,European Commitee of the Regions Members,25,0,0
10,US CIA World Leaders,12,0,0
0,Belgium Members of the Chamber of Representatives,2,0,0
3,Foreign Representatives in Canada: Heads of Mi...,2,0,0
8,Netherlands Senate,2,1,0



Highest-priority fuzzy review cases:


,opensanctions_id,opensanctions_primary_name,dataset,best_fuzzy_score,best_fuzzy_benchmark_person_name,best_fuzzy_benchmark_categories,fuzzy_review_priority,preliminary_review_hint
185,Q117235809,Meryem Kılıç-Karaaslan,Netherlands Senate;Wikidata Persons in Relevan...,95.00,Meryem Karaaslan,senate_members,high_priority_review,"Possible historical member, source-timing diff..."
1289,Q2279573,Henk Bakker,PEP position annotations by OpenSanctions;Wiki...,95.00,Henk Jan Bakker,ambassadors,high_priority_review,"Review role, timing, and empirical-scope eligi..."
2042,Q3074664,Jan de Ruiter,PEP position annotations by OpenSanctions;Wiki...,95.00,Klaas-Jan de Ruiter,registered_party_board,medium_priority_review,"Review role, timing, and empirical-scope eligi..."
2182,Q4446282,Ans van den Berg,PEP position annotations by OpenSanctions;Wiki...,88.24,Arjen van den Berg,ambassadors,medium_priority_review,"Review role, timing, and empirical-scope eligi..."
1259,Q2252298,Patrick van den Brink,PEP position annotations by OpenSanctions;Wiki...,88.21,V. van den Brink,high_judiciary_hr,medium_priority_review,"Review role, timing, and empirical-scope eligi..."
53,Q106044272,Heidi Lascaris-Bouhlel,Wikidata Persons in Relevant Categories;Wikida...,95.00,Heidi Bouhlel-Lascaris,registered_party_board,standard_review,"Review role, timing, and empirical-scope eligi..."
1595,Q25261995,Marianne de Jong-Meijer,PEP position annotations by OpenSanctions;Wiki...,95.00,Marianne de Jong,ambassadors,standard_review,"Review role, timing, and empirical-scope eligi..."
1646,Q2598765,Willem van Beek,PEP position annotations by OpenSanctions;Wiki...,92.86,Willem van Ee,ambassadors,standard_review,"Review role, timing, and empirical-scope eligi..."
2507,Q800091,Martin van Rijn,PEP position annotations by OpenSanctions;Wiki...,90.91,Martin van Rooijen,senate_members,standard_review,"Review role, timing, and empirical-scope eligi..."
8,Q101623881,Caroline van de Pol,PEP position annotations by OpenSanctions;Wiki...,90.00,Caroline van der Plas,house_members,standard_review,"Review role, timing, and empirical-scope eligi..."



Reverse-validation overview:
NL OpenSanctions entities: 2893
Linked through deterministic matching: 291
Unlinked after deterministic matching: 2602


## 5. Dataset-level profiling of unmatched Netherlands-assigned entities

The unmatched Netherlands-assigned OpenSanctions population is too large for exhaustive record-level review within the scope of this thesis.

This section profiles unmatched entities by source dataset and creates a transparent stratified review sample. The purpose is to identify whether unmatched records predominantly reflect excluded categories, historical records, local or subnational functions, state-owned-enterprise leadership, or potential omissions in the official-source benchmark.

In [7]:
# -------------------------------------------------------------------
# Create a prioritised dataset profile and stratified review sample
# -------------------------------------------------------------------

REVERSE_DATASET_PROFILE_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_dataset_profile.csv"
)

REVERSE_STRATIFIED_SAMPLE_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_stratified_review_sample.csv"
)

REVERSE_SCOPE_REVIEW_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_possible_scope_overlap_review.csv"
)


def classify_dataset_review_theme(dataset_value):
    """
    Assign a broad, non-final review theme from the OpenSanctions
    dataset label.

    This supports prioritisation only. It does not determine whether a
    record should have been included in the benchmark.
    """
    dataset_text = (
        ""
        if pd.isna(dataset_value)
        else str(dataset_value).lower()
    )

    if "state-owned enterprise" in dataset_text:
        return "state_owned_enterprise_or_public_company"

    if (
        "local government" in dataset_text
        or "municipal" in dataset_text
        or "province" in dataset_text
    ):
        return "subnational_or_local_government"

    if "senate" in dataset_text:
        return "senate_or_historical_senate_member"

    if (
        "house of representatives" in dataset_text
        or "everypolitician" in dataset_text
    ):
        return "parliamentary_or_historical_legislative_record"

    if (
        "court" in dataset_text
        or "judiciary" in dataset_text
        or "justice" in dataset_text
    ):
        return "judicial_or_legal_function"

    if (
        "political" in dataset_text
        or "party" in dataset_text
    ):
        return "political_party_or_public_office"

    if (
        "government" in dataset_text
        or "ministry" in dataset_text
        or "cabinet" in dataset_text
    ):
        return "government_or_public_administration"

    if "sanction" in dataset_text:
        return "sanctions_or_other_non_pep_source"

    if "wikidata" in dataset_text:
        return "wikidata_or_general_reference_source"

    return "other_or_unclear_source"


# -------------------------------------------------------------------
# Split multi-value OpenSanctions dataset labels into memberships
# -------------------------------------------------------------------

unmatched_dataset_profile_df = unmatched_nl_os_df[
    [
        "opensanctions_id",
        "opensanctions_primary_name",
        "dataset",
        "last_seen",
        "best_fuzzy_score",
        "best_fuzzy_benchmark_person_name",
        "best_fuzzy_benchmark_categories",
        "fuzzy_review_priority"
    ]
].copy()

unmatched_dataset_profile_df["dataset_label"] = (
    unmatched_dataset_profile_df["dataset"]
    .fillna("Unknown dataset")
    .astype(str)
    .str.split(";")
)

unmatched_dataset_profile_df = (
    unmatched_dataset_profile_df
    .explode("dataset_label")
    .copy()
)

unmatched_dataset_profile_df["dataset_label"] = (
    unmatched_dataset_profile_df["dataset_label"]
    .astype(str)
    .str.strip()
)

unmatched_dataset_profile_df["review_theme"] = (
    unmatched_dataset_profile_df["dataset_label"]
    .apply(classify_dataset_review_theme)
)

dataset_profile_summary_df = (
    unmatched_dataset_profile_df
    .groupby(
        [
            "review_theme",
            "dataset_label"
        ]
    )
    .agg(
        unmatched_entities=(
            "opensanctions_id",
            "nunique"
        ),
        earliest_last_seen=(
            "last_seen",
            "min"
        ),
        latest_last_seen=(
            "last_seen",
            "max"
        ),
        high_priority_fuzzy_candidates=(
            "fuzzy_review_priority",
            lambda values: (
                values
                .eq("high_priority_review")
                .sum()
            )
        ),
        medium_priority_fuzzy_candidates=(
            "fuzzy_review_priority",
            lambda values: (
                values
                .eq("medium_priority_review")
                .sum()
            )
        )
    )
    .reset_index()
    .sort_values(
        "unmatched_entities",
        ascending=False
    )
    .reset_index(drop=True)
)

dataset_theme_summary_df = (
    dataset_profile_summary_df
    .groupby("review_theme")
    .agg(
        datasets=(
            "dataset_label",
            "nunique"
        ),
        unmatched_entities=(
            "unmatched_entities",
            "sum"
        ),
        high_priority_fuzzy_candidates=(
            "high_priority_fuzzy_candidates",
            "sum"
        ),
        medium_priority_fuzzy_candidates=(
            "medium_priority_fuzzy_candidates",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "unmatched_entities",
        ascending=False
    )
    .reset_index(drop=True)
)

# -------------------------------------------------------------------
# Create a small reproducible review sample from major source datasets
# -------------------------------------------------------------------
# We take up to five records from each of the ten largest source
# datasets. This gives a manageable and transparent manual inspection
# set without claiming exhaustive validation.

top_dataset_labels = (
    dataset_profile_summary_df[
        "dataset_label"
    ]
    .head(10)
    .tolist()
)

stratified_review_sample_df = (
    unmatched_dataset_profile_df[
        unmatched_dataset_profile_df["dataset_label"]
        .isin(top_dataset_labels)
    ]
    .sort_values(
        [
            "dataset_label",
            "best_fuzzy_score",
            "opensanctions_primary_name"
        ],
        ascending=[True, False, True]
    )
    .groupby(
        "dataset_label",
        group_keys=False
    )
    .head(5)
    .copy()
)

stratified_review_sample_df["manual_classification"] = (
    "needs_manual_review"
)

stratified_review_sample_df["manual_review_notes"] = pd.NA

# -------------------------------------------------------------------
# Create a smaller subset for potential scope overlap review
# -------------------------------------------------------------------
# These are source themes most likely to reveal either excluded
# categories or possible benchmark gaps.

scope_overlap_themes = [
    "state_owned_enterprise_or_public_company",
    "judicial_or_legal_function",
    "government_or_public_administration",
    "political_party_or_public_office",
    "senate_or_historical_senate_member",
    "parliamentary_or_historical_legislative_record"
]

possible_scope_overlap_df = (
    unmatched_dataset_profile_df[
        unmatched_dataset_profile_df["review_theme"]
        .isin(scope_overlap_themes)
    ]
    .sort_values(
        [
            "review_theme",
            "best_fuzzy_score",
            "opensanctions_primary_name"
        ],
        ascending=[True, False, True]
    )
    .copy()
)

possible_scope_overlap_df["manual_classification"] = (
    "needs_manual_review"
)

possible_scope_overlap_df["manual_scope_decision"] = pd.NA
possible_scope_overlap_df["manual_review_notes"] = pd.NA

# -------------------------------------------------------------------
# Save outputs
# -------------------------------------------------------------------

write_csv_safely(
    dataset_profile_summary_df,
    REVERSE_DATASET_PROFILE_PATH
)

write_csv_safely(
    stratified_review_sample_df,
    REVERSE_STRATIFIED_SAMPLE_PATH
)

write_csv_safely(
    possible_scope_overlap_df,
    REVERSE_SCOPE_REVIEW_PATH
)

print("Unmatched entities by broad review theme:")
display(dataset_theme_summary_df)

print("\nTop 15 unmatched source datasets:")
display(dataset_profile_summary_df.head(15))

print("\nStratified manual review sample:")
display(
    stratified_review_sample_df[
        [
            "dataset_label",
            "review_theme",
            "opensanctions_primary_name",
            "last_seen",
            "best_fuzzy_score",
            "best_fuzzy_benchmark_person_name",
            "fuzzy_review_priority",
            "manual_classification"
        ]
    ]
)

print("\nPotential scope-overlap records:")
print(len(possible_scope_overlap_df))

print("\nSaved outputs:")
print(REVERSE_DATASET_PROFILE_PATH)
print(REVERSE_STRATIFIED_SAMPLE_PATH)
print(REVERSE_SCOPE_REVIEW_PATH)

Unmatched entities by broad review theme:


,review_theme,datasets,unmatched_entities,high_priority_fuzzy_candidates,medium_priority_fuzzy_candidates
0,wikidata_or_general_reference_source,1,2479,2,3
1,political_party_or_public_office,1,2117,2,3
2,sanctions_or_other_non_pep_source,1,1613,1,3
3,parliamentary_or_historical_legislative_record,1,248,0,0
4,other_or_unclear_source,8,76,0,0
5,senate_or_historical_senate_member,1,2,1,0



Top 15 unmatched source datasets:


,review_theme,dataset_label,unmatched_entities,earliest_last_seen,latest_last_seen,high_priority_fuzzy_candidates,medium_priority_fuzzy_candidates
0,wikidata_or_general_reference_source,Wikidata Persons in Relevant Categories,2479,2026-06-03T00:12:12,2026-06-04T14:31:22,2,3
1,political_party_or_public_office,Wikidata Politically Exposed Persons,2117,2026-06-04T08:46:59,2026-06-04T14:31:22,2,3
2,sanctions_or_other_non_pep_source,PEP position annotations by OpenSanctions,1613,2026-06-04T14:31:22,2026-06-04T14:31:22,1,3
3,parliamentary_or_historical_legislative_record,MySociety's EveryPolitician Legislators,248,2026-06-04T14:31:22,2026-06-04T14:31:22,0,0
4,other_or_unclear_source,European Parliament Members,32,2026-06-04T14:31:22,2026-06-04T14:31:22,0,0
5,other_or_unclear_source,European Commitee of the Regions Members,25,2026-06-04T14:31:22,2026-06-04T14:31:22,0,0
6,other_or_unclear_source,US CIA World Leaders,12,2026-06-04T14:31:22,2026-06-04T14:31:22,0,0
7,other_or_unclear_source,Belgium Members of the Chamber of Representatives,2,2026-06-04T14:31:22,2026-06-04T14:31:22,0,0
8,other_or_unclear_source,Foreign Representatives in Canada: Heads of Mi...,2,2026-05-28T19:43:01,2026-06-04T14:31:22,0,0
9,senate_or_historical_senate_member,Netherlands Senate,2,2026-06-04T08:46:59,2026-06-04T08:46:59,1,0



Stratified manual review sample:


,dataset_label,review_theme,opensanctions_primary_name,last_seen,best_fuzzy_score,best_fuzzy_benchmark_person_name,fuzzy_review_priority,manual_classification
2424,Belgium Members of the Chamber of Representatives,other_or_unclear_source,Melissa Depraetere,2026-06-04T14:31:22,60.61,Lisa Westerveld,standard_review,needs_manual_review
1246,Belgium Members of the Chamber of Representatives,other_or_unclear_source,Zuhal Demir,2026-06-04T14:31:22,60.35,Hanneke van der Werf,standard_review,needs_manual_review
253,European Commitee of the Regions Members,other_or_unclear_source,Ans MOL,2026-06-04T14:31:22,85.50,Jennes de Mol,standard_review,needs_manual_review
2598,European Commitee of the Regions Members,other_or_unclear_source,Arjen VAN GIJSSEL,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
2361,European Commitee of the Regions Members,other_or_unclear_source,Arthur van Dijk,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
2596,European Commitee of the Regions Members,other_or_unclear_source,Egbert VAN DIJK,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
1489,European Commitee of the Regions Members,other_or_unclear_source,Ellen van Selm,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
352,European Parliament Members,other_or_unclear_source,Anouk van Brug,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
2535,European Parliament Members,other_or_unclear_source,Bart Groothuis,2026-06-04T14:31:22,85.50,B.J. (Bart Jan) van Ettekoven,standard_review,needs_manual_review
353,European Parliament Members,other_or_unclear_source,Brigitte van den Berg,2026-06-04T14:31:22,85.50,A.J. van Doesum,standard_review,needs_manual_review



Potential scope-overlap records:
2367

Saved outputs:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_dataset_profile.csv
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_stratified_review_sample.csv
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_possible_scope_overlap_review.csv


## 6. Focused review of structured source memberships and excluded categories

OpenSanctions entities can have multiple dataset memberships. Broad reference datasets, such as Wikidata and general PEP annotations, are therefore excluded from this focused profile.

This section identifies unmatched Netherlands-assigned entities that occur in more specific source datasets, with particular attention to state-owned-enterprise leadership and other categories outside the empirical benchmark scope.

The output supports a limited manual review of structured-source records. It does not estimate benchmark completeness statistically.

In [8]:
# -------------------------------------------------------------------
# Focused source-membership profile for manual reverse validation
# -------------------------------------------------------------------
# Broad reference memberships are excluded here because they overlap
# heavily with many other source datasets and are not informative about
# the specific PEP function or source category.

FOCUSED_REVERSE_PROFILE_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_focused_source_profile.csv"
)

FOCUSED_REVIEW_SAMPLE_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_focused_review_sample.csv"
)

GENERIC_DATASET_LABELS = {
    "Wikidata Persons in Relevant Categories",
    "Wikidata Politically Exposed Persons",
    "PEP position annotations by OpenSanctions",
    "US CIA World Leaders"
}

focused_memberships_df = unmatched_dataset_profile_df[
    ~unmatched_dataset_profile_df["dataset_label"]
    .isin(GENERIC_DATASET_LABELS)
].copy()

print(
    "Unmatched source memberships after excluding broad "
    "reference datasets:",
    len(focused_memberships_df)
)

focused_dataset_profile_df = (
    focused_memberships_df
    .groupby("dataset_label")
    .agg(
        unmatched_nl_entities=(
            "opensanctions_id",
            "nunique"
        ),
        example_names=(
            "opensanctions_primary_name",
            lambda values: "; ".join(
                sorted(set(values))[:5]
            )
        ),
        earliest_last_seen=(
            "last_seen",
            "min"
        ),
        latest_last_seen=(
            "last_seen",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        "unmatched_nl_entities",
        ascending=False
    )
    .reset_index(drop=True)
)

# -------------------------------------------------------------------
# Explicitly identify likely Category H / SOE-related memberships
# -------------------------------------------------------------------

focused_dataset_profile_df["possible_scope_explanation"] = "other"

focused_dataset_profile_df.loc[
    focused_dataset_profile_df["dataset_label"]
    .str.contains(
        r"state-owned|state owned|enterprise|public company",
        case=False,
        na=False,
        regex=True
    ),
    "possible_scope_explanation"
] = "possible_excluded_state_owned_enterprise_category"

focused_dataset_profile_df.loc[
    focused_dataset_profile_df["dataset_label"]
    .str.contains(
        r"european parliament|committee of the regions",
        case=False,
        na=False,
        regex=True
    ),
    "possible_scope_explanation"
] = "eu_level_or_subnational_representative_function"

focused_dataset_profile_df.loc[
    focused_dataset_profile_df["dataset_label"]
    .str.contains(
        r"senate|house of representatives|everypolitician",
        case=False,
        na=False,
        regex=True
    ),
    "possible_scope_explanation"
] = "possible_historical_or_name_variation_case"

print("\nFocused unmatched source datasets:")
display(focused_dataset_profile_df.head(30))

print("\nPotential Category H / state-owned-enterprise datasets:")
display(
    focused_dataset_profile_df[
        focused_dataset_profile_df[
            "possible_scope_explanation"
        ]
        .eq(
            "possible_excluded_state_owned_enterprise_category"
        )
    ]
)

# -------------------------------------------------------------------
# Create a manageable review sample: up to five entities per source
# dataset among the ten largest focused datasets.
# -------------------------------------------------------------------

top_focused_datasets = (
    focused_dataset_profile_df["dataset_label"]
    .head(10)
    .tolist()
)

focused_review_sample_df = (
    focused_memberships_df[
        focused_memberships_df["dataset_label"]
        .isin(top_focused_datasets)
    ]
    .sort_values(
        [
            "dataset_label",
            "best_fuzzy_score",
            "opensanctions_primary_name"
        ],
        ascending=[True, False, True]
    )
    .groupby(
        "dataset_label",
        group_keys=False
    )
    .head(5)
    .copy()
)

focused_review_sample_df["manual_classification"] = (
    "needs_manual_review"
)

focused_review_sample_df["manual_scope_assessment"] = pd.NA
focused_review_sample_df["manual_review_notes"] = pd.NA

write_csv_safely(
    focused_dataset_profile_df,
    FOCUSED_REVERSE_PROFILE_PATH
)

write_csv_safely(
    focused_review_sample_df,
    FOCUSED_REVIEW_SAMPLE_PATH
)

print("\nFocused stratified review sample:")
display(
    focused_review_sample_df[
        [
            "dataset_label",
            "opensanctions_primary_name",
            "last_seen",
            "best_fuzzy_score",
            "best_fuzzy_benchmark_person_name",
            "fuzzy_review_priority",
            "manual_classification"
        ]
    ]
)

print("\nSaved focused source profile:")
print(FOCUSED_REVERSE_PROFILE_PATH)

print("\nSaved focused review sample:")
print(FOCUSED_REVIEW_SAMPLE_PATH)

Unmatched source memberships after excluding broad reference datasets: 314

Focused unmatched source datasets:


,dataset_label,unmatched_nl_entities,example_names,earliest_last_seen,latest_last_seen,possible_scope_explanation
0,MySociety's EveryPolitician Legislators,248,Achraf Bouali; Ady Thijsen; Agnes Mulder; Agne...,2026-06-04T14:31:22,2026-06-04T14:31:22,possible_historical_or_name_variation_case
1,European Parliament Members,32,Anja Hazekamp; Anna Strolenberg; Anouk van Bru...,2026-06-04T14:31:22,2026-06-04T14:31:22,eu_level_or_subnational_representative_function
2,European Commitee of the Regions Members,25,Ans MOL; Arjen VAN GIJSSEL; Arthur van Dijk; B...,2026-06-04T14:31:22,2026-06-04T14:31:22,other
3,Belgium Members of the Chamber of Representatives,2,Melissa Depraetere; Zuhal Demir,2026-06-04T14:31:22,2026-06-04T14:31:22,other
4,Foreign Representatives in Canada: Heads of Mi...,2,Grietje LANDMAN (MARGRIET VONNO); Martin Julia...,2026-05-28T19:43:01,2026-06-04T14:31:22,other
5,Netherlands Senate,2,Meryem Kılıç-Karaaslan; Ton van Kesteren,2026-06-04T08:46:59,2026-06-04T08:46:59,possible_historical_or_name_variation_case
6,French National Assembly,1,Julie Laernoes,2026-06-04T14:31:22,2026-06-04T14:31:22,other
7,Germany Members of the Bundestag,1,Marc Jongen,2026-06-04T14:31:22,2026-06-04T14:31:22,other
8,Germany Members of the Bundesrat,1,Rudi Hoogvliet,2026-06-04T08:46:59,2026-06-04T08:46:59,other



Potential Category H / state-owned-enterprise datasets:


,dataset_label,unmatched_nl_entities,example_names,earliest_last_seen,latest_last_seen,possible_scope_explanation



Focused stratified review sample:


,dataset_label,opensanctions_primary_name,last_seen,best_fuzzy_score,best_fuzzy_benchmark_person_name,fuzzy_review_priority,manual_classification
2424,Belgium Members of the Chamber of Representatives,Melissa Depraetere,2026-06-04T14:31:22,60.61,Lisa Westerveld,standard_review,needs_manual_review
1246,Belgium Members of the Chamber of Representatives,Zuhal Demir,2026-06-04T14:31:22,60.35,Hanneke van der Werf,standard_review,needs_manual_review
253,European Commitee of the Regions Members,Ans MOL,2026-06-04T14:31:22,85.50,Jennes de Mol,standard_review,needs_manual_review
2598,European Commitee of the Regions Members,Arjen VAN GIJSSEL,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
2361,European Commitee of the Regions Members,Arthur van Dijk,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
2596,European Commitee of the Regions Members,Egbert VAN DIJK,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
1489,European Commitee of the Regions Members,Ellen van Selm,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
352,European Parliament Members,Anouk van Brug,2026-06-04T14:31:22,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
2535,European Parliament Members,Bart Groothuis,2026-06-04T14:31:22,85.50,B.J. (Bart Jan) van Ettekoven,standard_review,needs_manual_review
353,European Parliament Members,Brigitte van den Berg,2026-06-04T14:31:22,85.50,A.J. van Doesum,standard_review,needs_manual_review



Saved focused source profile:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_focused_source_profile.csv

Saved focused review sample:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_focused_review_sample.csv


## 7. Entity-level scope diagnostic for unmatched OpenSanctions records

OpenSanctions entities may occur in multiple source datasets. Dataset-level counts are therefore overlapping and cannot be added to estimate the number of unmatched persons.

This section assigns each unmatched Netherlands-assigned OpenSanctions entity to one mutually exclusive diagnostic group based on its available source memberships. The purpose is to distinguish broad reference-only records from records associated with EU-level functions, foreign offices, Dutch national institutions, or historical legislative sources.

These classifications are diagnostic heuristics and do not constitute a statistical estimate of benchmark completeness.

In [15]:
# -------------------------------------------------------------------
# Entity-level diagnostic classification of unmatched NL entities
# -------------------------------------------------------------------
# This avoids double-counting entities that occur in several
# OpenSanctions source datasets.

ENTITY_SCOPE_DIAGNOSTIC_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_entity_scope_diagnostic.csv"
)

ENTITY_SCOPE_SUMMARY_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_entity_scope_summary.csv"
)

ENTITY_SCOPE_REVIEW_SAMPLE_PATH = (
    OUTPUT_DIR
    / "opensanctions_nl_unmatched_entity_scope_review_sample.csv"
)


GENERIC_DATASET_LABELS = {
    "Wikidata Persons in Relevant Categories",
    "Wikidata Politically Exposed Persons",
    "PEP position annotations by OpenSanctions",
    "US CIA World Leaders"
}


def parse_dataset_memberships(dataset_value):
    """
    Convert one OpenSanctions dataset field into a clean list of
    dataset memberships.

    The export may contain one dataset, or several datasets separated
    by semicolons or vertical bars.
    """
    if pd.isna(dataset_value):
        return []

    dataset_text = str(dataset_value).strip()

    if dataset_text == "":
        return []

    return sorted(
        {
            label.strip()
            for label in re.split(
                r"[;|]",
                dataset_text
            )
            if label.strip()
        }
    )


def classify_unmatched_entity_scope(dataset_memberships):
    """
    Assign one mutually exclusive diagnostic classification.

    The classification is based on source memberships only. It is a
    review aid, not a final substantive judgement about a person's
    PEP status or benchmark inclusion.
    """
    non_generic_memberships = [
        label
        for label in dataset_memberships
        if label not in GENERIC_DATASET_LABELS
    ]

    source_text = " | ".join(
        non_generic_memberships
    ).casefold()

    # No specific source remains after generic reference sources have
    # been removed.
    if len(non_generic_memberships) == 0:
        return "generic_reference_only"

    # Dutch national institutions receive priority because these are
    # the records most relevant for a possible timing/name review.
    if (
        "netherlands senate" in source_text
        or "netherlands house of representatives" in source_text
    ):
        return "dutch_national_institution_review"

    # EveryPolitician may include historical or former legislators and
    # should not automatically be interpreted as a current omission.
    if "everypolitician" in source_text:
        return "historical_or_name_variation_review"

    # These sources represent EU-level or subnational functions that
    # were not part of the national Dutch benchmark scope.
    if (
    "european parliament" in source_text
    or "committee of the regions" in source_text
    or "commitee of the regions" in source_text
    ):
        return "eu_or_subnational_function_out_of_scope"

    # Foreign legislative and diplomatic records are not evidence of a
    # missing Dutch national benchmark record.
    if any(
        country_term in source_text
        for country_term in [
            "belgium",
            "french",
            "france",
            "germany",
            "bundestag",
            "bundesrat",
            "canada"
        ]
    ):
        return "foreign_office_outside_dutch_scope"

    return "other_structured_source_review"


# -------------------------------------------------------------------
# Create one row per unmatched OpenSanctions entity
# -------------------------------------------------------------------

entity_scope_diagnostic_df = unmatched_nl_os_df[
    [
        "opensanctions_id",
        "opensanctions_primary_name",
        "countries",
        "dataset",
        "last_seen",
        "best_fuzzy_score",
        "best_fuzzy_benchmark_person_name",
        "best_fuzzy_benchmark_categories",
        "fuzzy_review_priority"
    ]
].copy()

entity_scope_diagnostic_df["dataset_memberships"] = (
    entity_scope_diagnostic_df["dataset"]
    .apply(parse_dataset_memberships)
)

entity_scope_diagnostic_df["non_generic_dataset_memberships"] = (
    entity_scope_diagnostic_df["dataset_memberships"]
    .apply(
        lambda memberships: [
            label
            for label in memberships
            if label not in GENERIC_DATASET_LABELS
        ]
    )
)

entity_scope_diagnostic_df[
    "non_generic_dataset_count"
] = (
    entity_scope_diagnostic_df[
        "non_generic_dataset_memberships"
    ]
    .apply(len)
)

entity_scope_diagnostic_df[
    "entity_scope_classification"
] = (
    entity_scope_diagnostic_df[
        "dataset_memberships"
    ]
    .apply(classify_unmatched_entity_scope)
)

entity_scope_diagnostic_df[
    "non_generic_dataset_memberships"
] = (
    entity_scope_diagnostic_df[
        "non_generic_dataset_memberships"
    ]
    .apply(
        lambda memberships: "; ".join(memberships)
    )
)

# -------------------------------------------------------------------
# Summarise mutually exclusive entity-level classifications
# -------------------------------------------------------------------

entity_scope_summary_df = (
    entity_scope_diagnostic_df
    .groupby("entity_scope_classification")
    .agg(
        unmatched_entities=(
            "opensanctions_id",
            "nunique"
        ),
        high_priority_fuzzy_candidates=(
            "fuzzy_review_priority",
            lambda values: (
                values
                .eq("high_priority_review")
                .sum()
            )
        ),
        medium_priority_fuzzy_candidates=(
            "fuzzy_review_priority",
            lambda values: (
                values
                .eq("medium_priority_review")
                .sum()
            )
        ),
        example_names=(
            "opensanctions_primary_name",
            lambda values: "; ".join(
                sorted(
                    set(values)
                )[:5]
            )
        )
    )
    .reset_index()
    .sort_values(
        "unmatched_entities",
        ascending=False
    )
    .reset_index(drop=True)
)

# -------------------------------------------------------------------
# Prepare a manageable manual-review output
# -------------------------------------------------------------------
# Review every Dutch-institution record because this group should be
# small. For the larger historical/name-variation group, create a
# reproducible random sample of 15 entities.

dutch_institution_review_df = (
    entity_scope_diagnostic_df[
        entity_scope_diagnostic_df[
            "entity_scope_classification"
        ]
        .eq("dutch_national_institution_review")
    ]
    .copy()
)

historical_review_pool_df = (
    entity_scope_diagnostic_df[
        entity_scope_diagnostic_df[
            "entity_scope_classification"
        ]
        .eq("historical_or_name_variation_review")
    ]
    .copy()
)

historical_sample_size = min(
    15,
    len(historical_review_pool_df)
)

if historical_sample_size > 0:
    historical_review_sample_df = (
        historical_review_pool_df
        .sample(
            n=historical_sample_size,
            random_state=42
        )
        .copy()
    )
else:
    historical_review_sample_df = (
        historical_review_pool_df.copy()
    )

dutch_institution_review_df["review_selection"] = (
    "all_dutch_national_institution_records"
)

historical_review_sample_df["review_selection"] = (
    "reproducible_random_sample"
)

# Add every high- or medium-priority fuzzy candidate, regardless of
# source-membership classification. These are few enough for review
# and may identify unresolved name variations.

priority_fuzzy_review_df = (
    entity_scope_diagnostic_df[
        entity_scope_diagnostic_df[
            "fuzzy_review_priority"
        ]
        .isin(
            [
                "high_priority_review",
                "medium_priority_review"
            ]
        )
    ]
    .copy()
)

# Do not duplicate records already selected because they belong to a
# Dutch national institution or the EveryPolitician sample.
priority_fuzzy_review_df = (
    priority_fuzzy_review_df[
        ~priority_fuzzy_review_df[
            "opensanctions_id"
        ]
        .isin(
            pd.concat(
                [
                    dutch_institution_review_df[
                        "opensanctions_id"
                    ],
                    historical_review_sample_df[
                        "opensanctions_id"
                    ]
                ]
            )
        )
    ]
    .copy()
)

priority_fuzzy_review_df["review_selection"] = (
    "all_high_medium_fuzzy_candidates"
)

entity_scope_review_sample_df = pd.concat(
    [
        dutch_institution_review_df,
        historical_review_sample_df,
        priority_fuzzy_review_df
    ],
    ignore_index=True
)

entity_scope_review_sample_df = (
    entity_scope_review_sample_df
    .drop_duplicates(
        subset=["opensanctions_id"]
    )
    .reset_index(drop=True)
)

entity_scope_review_sample_df["manual_classification"] = (
    "needs_manual_review"
)

entity_scope_review_sample_df["manual_review_notes"] = pd.NA

# -------------------------------------------------------------------
# Save outputs
# -------------------------------------------------------------------

write_csv_safely(
    entity_scope_diagnostic_df,
    ENTITY_SCOPE_DIAGNOSTIC_PATH
)

write_csv_safely(
    entity_scope_summary_df,
    ENTITY_SCOPE_SUMMARY_PATH
)

write_csv_safely(
    entity_scope_review_sample_df,
    ENTITY_SCOPE_REVIEW_SAMPLE_PATH
)

print("Mutually exclusive entity-level scope diagnostic:")
display(entity_scope_summary_df)

print("\nRecords selected for limited manual review:")
display(
    entity_scope_review_sample_df[
        [
            "review_selection",
            "entity_scope_classification",
            "opensanctions_primary_name",
            "non_generic_dataset_memberships",
            "best_fuzzy_score",
            "best_fuzzy_benchmark_person_name",
            "fuzzy_review_priority",
            "manual_classification"
        ]
    ]
)

print("\nSaved outputs:")
print(ENTITY_SCOPE_DIAGNOSTIC_PATH)
print(ENTITY_SCOPE_SUMMARY_PATH)
print(ENTITY_SCOPE_REVIEW_SAMPLE_PATH)

Mutually exclusive entity-level scope diagnostic:


,entity_scope_classification,unmatched_entities,high_priority_fuzzy_candidates,medium_priority_fuzzy_candidates,example_names
0,generic_reference_only,2293,1,3,A. Schout; A.P.J. van Bastelaar; Aad Kosto; Aa...
1,historical_or_name_variation_review,248,0,0,Achraf Bouali; Ady Thijsen; Agnes Mulder; Agne...
2,eu_or_subnational_function_out_of_scope,54,0,0,Anja Hazekamp; Anna Strolenberg; Anouk van Bru...
3,foreign_office_outside_dutch_scope,5,0,0,Grietje LANDMAN (MARGRIET VONNO); Julie Laerno...
4,dutch_national_institution_review,2,1,0,Meryem Kılıç-Karaaslan; Ton van Kesteren



Records selected for limited manual review:


,review_selection,entity_scope_classification,opensanctions_primary_name,non_generic_dataset_memberships,best_fuzzy_score,best_fuzzy_benchmark_person_name,fuzzy_review_priority,manual_classification
0,all_dutch_national_institution_records,dutch_national_institution_review,Meryem Kılıç-Karaaslan,Netherlands Senate,95.00,Meryem Karaaslan,high_priority_review,needs_manual_review
1,all_dutch_national_institution_records,dutch_national_institution_review,Ton van Kesteren,Netherlands Senate,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review
2,reproducible_random_sample,historical_or_name_variation_review,Ockje Tellegen,MySociety's EveryPolitician Legislators,85.50,A.J.C. de Moor-van Vugt,standard_review,needs_manual_review
3,reproducible_random_sample,historical_or_name_variation_review,Albert de Vries,MySociety's EveryPolitician Legislators,85.50,C. de Kruif **,standard_review,needs_manual_review
4,reproducible_random_sample,historical_or_name_variation_review,Frank Futselaar,MySociety's EveryPolitician Legislators,85.50,C.F.E. van Olden-Smit,standard_review,needs_manual_review
5,reproducible_random_sample,historical_or_name_variation_review,Paulus Jansen,MySociety's EveryPolitician Legislators,69.23,Paul van Gent,standard_review,needs_manual_review
6,reproducible_random_sample,historical_or_name_variation_review,Wim-Jan Renkema,MySociety's EveryPolitician Legislators,85.50,Wim Geerts,standard_review,needs_manual_review
7,reproducible_random_sample,historical_or_name_variation_review,Agnes Mulder,MySociety's EveryPolitician Legislators,85.50,A.E.B. ter Heide,standard_review,needs_manual_review
8,reproducible_random_sample,historical_or_name_variation_review,Gerrit Schotte,MySociety's EveryPolitician Legislators,85.50,E.F. Faase,standard_review,needs_manual_review
9,reproducible_random_sample,historical_or_name_variation_review,Evelyn Wever-Croes,MySociety's EveryPolitician Legislators,85.50,A.E.H. van der Voort Maarschalk,standard_review,needs_manual_review



Saved outputs:
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_entity_scope_diagnostic.csv
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_entity_scope_summary.csv
c:\Users\steph\Documents\Education\MSc Business Analytics\Thesis\Code_clean\data\output\opensanctions_nl_unmatched_entity_scope_review_sample.csv
